# Outflow boundaries & acoustic power: choked nozzle and metered bleed

Beyond the static-pressure outlet, Nefes has two flow-fixing outflow boundaries:

* **`mass_flow_outlet`** — pins the outflow rate; the static pressure floats. Its natural
  acoustic closure is **constant mass flow**, $\dot m' = 0$.
* **`choked_nozzle_outlet`** — a compact sonic throat of area $A^\*$ lumped just downstream.
  The outflow is the **critical mass flux** for the interior stagnation state, the approach
  plane stays subsonic, and its acoustic closure is the compact choked-nozzle
  (**Marble–Candel**) reflection, entropy coupling included.

A key property of Nefes: each boundary's acoustic closure is the *linearization of its mean
residual*. We never specify a separate acoustic BC in this rig — the `total_pressure_inlet`,
`choked_nozzle_outlet` and `mass_flow_outlet` all **inherit** their closures. Below we pull
each one straight out of the converged Jacobian $J_\text{alg}$ and check it against the
textbook formulas.

We then use the **acoustic-power diagnostics** (`nefes.perturbation.fields.power`) to read the energy
budget of the network. This matters because with mean flow a reflection coefficient $|R|>1$
does **not** mean energy is created, $|R|<1$ does not guarantee absorption, and — as we'll see
— the physically natural reservoir inlet turns out to be exactly *energy-neutral*. Deliberately
over-reflecting it is what (spuriously) destabilises an otherwise passive network.

In [ ]:
import os, sys, warnings

_root = os.getcwd()
while not os.path.isdir(os.path.join(_root, "nefes")) and _root != os.path.dirname(_root):
    _root = os.path.dirname(_root)
sys.path.insert(0, _root)

import numpy as np
import plotly.graph_objects as go

from nefes.elements import catalog as cat
from nefes.elements.ids import CHOKED_NOZZLE_OUTLET, MASS_FLOW_OUTLET, PT_INLET
from nefes.perturbation.operator.boundary_bc import PerturbationBC
from nefes.perturbation import eigenmodes, find_terminals
from nefes.perturbation import acoustic_intensity, acoustic_energy_density, passive_reflection_bound
from nefes.perturbation.operator.operator import build_acoustic_blocks
from nefes.perturbation.operator.characteristics import char_to_dx
from nefes.plotting import use_nefes_theme
from nefes.shell import Network
from nefes.solver import solve
from nefes.solver.control import states_table
from nefes.thermo.configure import perfect_gas
from nefes.assembly.derive import ES_MDOT, ES_M, ES_P, ES_RHO, ES_C, ES_U, ES_AREA

use_nefes_theme()
R, GAMMA = 287.0, 1.4
CP = GAMMA * R / (GAMMA - 1.0)
K = CP / R

## 1. A branched rig: choked core nozzle + metered bleed

A reservoir (total-pressure inlet) feeds a duct into a **splitter**; one branch exhausts
through a **choked nozzle** (throat $A^\*=0.015\,\mathrm{m^2}$), the other is a **metered
bleed** drawing a fixed $2\,\mathrm{kg/s}$:

```
reservoir(pt) ── feed ──[splitter]── core ── (choked nozzle A*)
                              └──── bleed ── (mass-flow outlet)
```

The reservoir pins the pressure level; the choke and the bleed each pin a branch flow, so
the inlet mass flow self-adjusts to supply both. Everything stays subsonic — the sonic
point lives inside the lumped nozzle throat, not in the domain.

In [ ]:
def build_rig(inlet_bc=None):
    "Branched rig. inlet_bc=None keeps the reservoir's inherited acoustic closure."
    els = [
        cat.total_pressure_inlet(2.5e5, 300.0, name="reservoir", perturbation_bc=inlet_bc),
        cat.duct(0.6, name="feed"),
        cat.splitter(name="manifold"),
        cat.duct(0.4, name="core"),
        cat.choked_nozzle_outlet(0.015, name="nozzle"),   # inherits Marble-Candel
        cat.duct(0.5, name="bleedpipe"),
        cat.mass_flow_outlet(2.0, name="bleed"),           # inherits mdot' = 0
    ]
    edges = [(0, 1, 0.05), (1, 2, 0.05), (2, 3, 0.03), (3, 4, 0.03), (2, 5, 0.02), (5, 6, 0.02)]
    prob = cat.build_problem(perfect_gas(R, GAMMA), els, edges, mdot_ref=6.0, p_ref=1.5e5, h_ref=CP * 300.0)
    network = Network(perfect_gas(R, GAMMA), p_ref=1.5e5, mdot_ref=6.0, h_ref=CP * 300.0, nodes=els, edges=edges)
    return prob, network

prob, network = build_rig()
res = solve(prob)
assert res.converged
est = states_table(prob, res.x)
print(f"reservoir feed : mdot = {est[ES_MDOT,1]:6.3f} kg/s   M = {est[ES_M,1]:.3f}   p = {est[ES_P,1]:8.0f} Pa")
print(f"choked core    : mdot = {est[ES_MDOT,3]:6.3f} kg/s   M = {est[ES_M,3]:.3f}   (sonic-throat critical flux)")
print(f"metered bleed  : mdot = {est[ES_MDOT,5]:6.3f} kg/s   M = {est[ES_M,5]:.3f}   (pinned)")
print(f"mass balance   : feed {est[ES_MDOT,1]:.3f}  =  nozzle {est[ES_MDOT,3]:.3f} + bleed {est[ES_MDOT,5]:.3f}")

## 2. Well-posedness: every case needs a pressure reference

The new flow-fixing outlets make it easy to write an **ill-posed** boundary set. If *every*
boundary only pins a mass flow (`mass_flow_inlet`, `mass_flow_outlet`, `wall`), the absolute
pressure level is a free gauge — adding a constant to every pressure leaves the residual
unchanged — so the steady Jacobian is singular. Nefes catches this **before solving**:

In [ ]:
def try_build(elements, edges):
    try:
        cat.build_problem(perfect_gas(R, GAMMA), elements, edges, mdot_ref=5.0, p_ref=1e5, h_ref=CP * 300.0)
        return "built OK"
    except ValueError as e:
        return f"REJECTED: {e}"

edges2 = [(0, 1, 0.05), (1, 2, 0.05)]
print("mass_flow_inlet + mass_flow_outlet:")
print("  ", try_build([cat.mass_flow_inlet(5.0, 300.0), cat.duct(0.4), cat.mass_flow_outlet(5.0)], edges2))
print("\nmass_flow_inlet + choked_nozzle_outlet:")
print("  ", try_build([cat.mass_flow_inlet(5.0, 300.0), cat.duct(0.4), cat.choked_nozzle_outlet(0.03)], edges2))

A `total_pressure_inlet`/`pressure_outlet` pins the level directly; a
`choked_nozzle_outlet` pins it through its critical-mass-flux relation. So
`mass_flow_inlet + choked_nozzle_outlet` is fine.

## 3. The acoustic closures come straight out of $J_\text{alg}$

The claim "inherited for free" is concrete: the converged complex-step Jacobian $J_\text{alg}$
*is* the zero-frequency perturbation operator, so each terminal's row in $J_\text{alg}$, rotated
into characteristic coordinates $(f,g,h)$, **is** its acoustic closure. We read all three:

* **choked nozzle** (a duct *head* / outlet): $c_f f + c_g g + c_h h = 0 \Rightarrow g = R f + R_s h$,
  with $R=-c_f/c_g$, $R_s=-c_h/c_g$ — compared against Marble–Candel.
* **mass-flow outlet**: the same read-out, compared against constant mass flow $\tfrac{1+M}{1-M}$.
* **reservoir** (a duct *tail* / inlet): here $f$ is specified from the arriving $g$. A stagnation
  reservoir fixes $p_t$ and $T_t$, so it seeds **no entropy wave** ($h_\text{in}=0$); the row then
  collapses to $R = f/g = -c_g/c_f$, which we will see is $-\tfrac{1-M}{1+M}$.

No hand-coded boundary condition was applied anywhere.

In [ ]:
blocks = build_acoustic_blocks(prob, res.x)
ns = int(prob.n_solve)
terms = {t.rid: t for t in find_terminals(prob, res.x)}

def closure_row(term):
    "The terminal's J_alg row, rotated into characteristic coordinates (c_f, c_g, c_h)."
    e = term.edge
    r0 = int(prob.node_row_ptr[term.node])
    row = np.asarray(blocks.J_alg[r0, ns * e : ns * e + 3].todense()).ravel()
    state = [float(est[k, e]) for k in (ES_RHO, ES_C, ES_U, ES_P, ES_AREA)]
    return row @ char_to_dx(*state, K)

# choked nozzle (outlet)  ->  Marble-Candel
tn = terms[CHOKED_NOZZLE_OUTLET]
Mn = float(est[ES_M, tn.edge]); rho_n, c_n = float(est[ES_RHO, tn.edge]), float(est[ES_C, tn.edge])
cf, cg, ch = closure_row(tn)
R_inh, Rs_inh = -cf / cg, -ch / cg
R_mc = (2 - (GAMMA - 1) * Mn) / (2 + (GAMMA - 1) * Mn)
Rs_mc = (c_n / rho_n) * Mn / (2 + (GAMMA - 1) * Mn)
print(f"choked nozzle    (outlet, M={Mn:.3f}):")
print(f"   R   : J_alg {R_inh.real:+.6f}   Marble-Candel {R_mc:+.6f}   |diff| {abs(R_inh - R_mc):.1e}")
print(f"   R_s : J_alg {Rs_inh.real:+.4f}   Marble-Candel {Rs_mc:+.4f}   |diff| {abs(Rs_inh - Rs_mc):.1e}")

# mass-flow outlet (outlet)  ->  constant mass flow
tb = terms[MASS_FLOW_OUTLET]
Mb = float(est[ES_M, tb.edge])
cf, cg, ch = closure_row(tb)
R_inh_b = -cf / cg
R_cmf_b = (1 + Mb) / (1 - Mb)
print(f"\nmass-flow outlet (outlet, M={Mb:.3f}):")
print(f"   R   : J_alg {R_inh_b.real:+.6f}   (1+M)/(1-M) {R_cmf_b:+.6f}   |diff| {abs(R_inh_b - R_cmf_b):.1e}")

# reservoir total-pressure inlet (inlet)  ->  convective open end (h_in = 0)
ti = terms[PT_INLET]
Mi = float(est[ES_M, ti.edge])
cf, cg, ch = closure_row(ti)
R_inh_i = -cg / cf          # inlet: f specified from the arriving g, with h_in = 0
R_open = -(1 - Mi) / (1 + Mi)
print(f"\nreservoir        (total-pressure inlet, M={Mi:.3f}):")
print(f"   R   : J_alg {R_inh_i.real:+.6f}   -(1-M)/(1+M) {R_open:+.6f}   |diff| {abs(R_inh_i - R_open):.1e}")
print(f"   (identical to mean_flow_open_end -- the convective open end)")

So all three closures fall out of the mean-flow Jacobian to round-off. Now plot the two
*outlet* reflection laws across approach Mach. Note where they sit relative to $R=1$ — and
hold that thought: both $\to 1$ as $M\to0$ (a slow approach looks rigid), the choked nozzle
drops *below* 1, and constant mass flow climbs *above* 1.

In [ ]:
M = np.linspace(0.0, 0.6, 121)
rho, c = 1.2, 340.0
R_choked = [PerturbationBC.choked_nozzle().reflection_coefficient(0.0, rho, c, m, K).real for m in M]
R_cmf = [PerturbationBC.constant_mass_flow().reflection_coefficient(0.0, rho, c, m).real for m in M]
fig = go.Figure()
fig.add_scatter(x=M, y=R_choked, mode="lines", name="choked nozzle (Marble-Candel)")
fig.add_scatter(x=M, y=R_cmf, mode="lines", name="constant mass flow  (mdot' = 0)")
fig.add_hline(y=1.0, line_dash="dot", annotation_text="rigid wall R = +1")
fig.update_layout(title="Inherited outlet reflection vs. approach Mach",
                  xaxis_title="approach Mach M", yaxis_title="reflection coefficient R", template="nefes")
fig.show()

## 4. Acoustic power with mean flow: the energy-neutral reflection bounds

With a moving mean flow, acoustic energy is **not** carried in proportion to $|p'|^2$. The
time-averaged energy flux of the down/up-running characteristics $f,g$ across a section is

$$ I \;=\; \tfrac12\rho c\big[(1+M)^2|f|^2 \;-\; (1-M)^2|g|^2\big]. $$

The mean flow biases the two directions by $(1\pm M)^2$: a downstream wave is a *stronger*
energy carrier than an upstream one of equal amplitude. This shifts the **energy-neutral**
(lossless) reflection magnitude away from 1 — to $\frac{1+M}{1-M}$ at an **outlet** and
$\frac{1-M}{1+M}$ at an **inlet**. Above that bound a terminal is an acoustic source; below it,
an absorber. All three of our inherited closures land on physically meaningful spots:

* **Constant mass flow, $R=\frac{1+M}{1-M}>1$, sits *on* the outlet bound — lossless.** The
  reflected wave needs amplitude $\frac{1+M}{1-M}$ just to carry back the *same power* it
  received: reflected/incident power $= R^2\frac{(1-M)^2}{(1+M)^2}=1$ exactly. A perfect mirror;
  $|R|>1$ is pure $(1\pm M)^2$ bookkeeping, not energy creation.
* **Choked nozzle, $R<1$, sits *below* it — absorbs.** Most of the incident acoustic power
  *leaves through the throat* (convected out past the sonic point — it is not a closed end). At
  the rig's $M\approx0.31$ only ~20% is reflected.
* **The inherited reservoir, $R=-\frac{1-M}{1+M}$, sits *on* the inlet bound — also
  energy-neutral.** Holding stagnation pressure constant does no net acoustic work; acoustically
  it is the convective open end. The reservoir neither feeds nor drains acoustic energy.

We plot the bounds with `passive_reflection_bound` and mark where the rig's reservoir falls:

In [ ]:
# the primitives in action: a unit f-wave's energy travels at u+c, a unit g-wave at u-c
m0, rho0, c0 = 0.3, 1.2, 340.0
v_fwd = acoustic_intensity(rho0, c0, m0, 1.0, 0.0) / acoustic_energy_density(rho0, m0, 1.0, 0.0)
v_bwd = acoustic_intensity(rho0, c0, m0, 0.0, 1.0) / acoustic_energy_density(rho0, m0, 0.0, 1.0)
print(f"energy group speed @ M={m0}:  forward {v_fwd:.1f} = u+c {c0*(1+m0):.1f}   "
      f"backward {v_bwd:.1f} = u-c {c0*(m0-1):.1f}")

Min = float(est[ES_M, 1])           # reservoir / feed Mach
neutral_out = [passive_reflection_bound(m, "outlet") for m in M]
neutral_in = [passive_reflection_bound(m, "inlet") for m in M]
fig = go.Figure()
fig.add_scatter(x=M, y=neutral_out, mode="lines", line=dict(color="#d62728", dash="dash"),
                name="outlet neutral |R| = (1+M)/(1-M)")
fig.add_scatter(x=M, y=R_cmf, mode="lines", line=dict(color="#2ca02c"),
                name="constant mass flow  (on outlet neutral: lossless)")
fig.add_scatter(x=M, y=R_choked, mode="lines", name="choked nozzle  (below: absorbs)")
fig.add_scatter(x=M, y=neutral_in, mode="lines", line=dict(color="#1f77b4", dash="dash"),
                name="inlet neutral |R| = (1-M)/(1+M)")
fig.add_scatter(x=[Min], y=[passive_reflection_bound(Min, "inlet")], mode="markers",
                marker=dict(size=11, color="#1f77b4"), name="inherited reservoir (on inlet neutral)")
fig.add_hline(y=0.8, line_dash="dot", annotation_text="an over-reflecting inlet |R| = 0.8  (section 5)")
fig.add_hline(y=1.0, line_dash="dot", line_color="#aaa")
fig.update_layout(
    title="Energy-neutral reflection bounds: |R| above the bound = acoustic source",
    xaxis_title="approach Mach M", yaxis_title="|R|", yaxis_range=[0, 2.2], template="nefes")
fig.show()
print(f"At the rig's inlet M = {Min:.3f}, the inherited reservoir |R| = "
      f"{passive_reflection_bound(Min,'inlet'):.3f} sits exactly on the energy-neutral inlet bound.")

## 5. The spectrum and the boundary power budget

With every terminal *on or below* its passivity bound, this rig has **no acoustic energy
source** — so every mode must decay. The acoustic-power diagnostic confirms it and shows
*where* the energy goes. For a self-sustained mode the global balance is
$dE/dt = 2\sigma E = \sum_\text{boundaries} W_\text{in}$, so the **sign of the net boundary
power equals the sign of the growth rate**, and the per-boundary breakdown says who absorbs.
`result.boundary_power(i)` computes the Myers flux through each terminal.

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    spec = eigenmodes(prob, res.x, freq_band=(50.0, 500.0), growth_band=(-150.0, 150.0), isentropic=True)
for i in range(spec.n_modes):
    m = spec.summary()[i]
    print(f"mode {i}:  f = {m['freq_hz']:7.2f} Hz    growth = {m['growth_rate']:+8.2f} 1/s    "
          f"{'GROWING' if m['unstable'] else 'decays'}")

bp = spec.boundary_power(0)
print()
print(bp.table())
spec.plot_spectrum(title="Branched rig with the inherited reservoir: every mode decays").show()

In [ ]:
bp.plot(title=f"Energy budget of the {bp.freq_hz:.0f} Hz mode  (red = source, blue = sink)").show()

The budget is unambiguous: the **reservoir contributes exactly $0\%$** (it sits on its
energy-neutral bound), the **mass-flow bleed is also $0\%$** (a lossless mirror), and the
**choked nozzle is the sole sink** — it carries away the acoustic power through its throat. Net
boundary power is negative for every mode, so the network is unconditionally stable. This is the
*physically correct* behaviour of a stagnation-reservoir-fed cold-flow rig.

### What an over-reflecting inlet does

Now override the reservoir with a hard partial reflection $|R|=0.8$ — *above* its
energy-neutral $0.64$. Nothing about the mean flow changes, but the inlet is now an acoustic
**source**, and it manufactures a growing mode out of nothing:

In [ ]:
prob_a, _ = build_rig(PerturbationBC.reflection(0.8))   # 0.8 > (1-M)/(1+M) = 0.64
res_a = solve(prob_a)
assert res_a.converged
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    spec_a = eigenmodes(prob_a, res_a.x, freq_band=(50.0, 500.0), growth_band=(-150.0, 150.0), isentropic=True)
for i in range(spec_a.n_modes):
    m = spec_a.summary()[i]
    flag = "  <-- growing" if m["unstable"] else ""
    print(f"mode {i}:  f = {m['freq_hz']:7.2f} Hz    growth = {m['growth_rate']:+8.2f} 1/s{flag}")

i_grow = int(np.argmax(spec_a.growth_rates))
bp_a = spec_a.boundary_power(i_grow)
print()
print(bp_a.table())
spec_a.plot_spectrum(title="Same rig, reservoir forced to |R| = 0.8: a spurious growing mode").show()

In [ ]:
bp_a.plot(title=f"Who drives the {bp_a.freq_hz:.0f} Hz mode? (red = source, blue = sink)").show()

The bar chart attributes the growth: the **reservoir is now the source** (reflecting at
$|R|=0.8>0.64$), the **choked nozzle remains the sink**, and the **bleed stays neutral** ($0\%$).
The net is positive — matching the positive growth rate. The "instability" is entirely an
artifact of an over-reflecting inlet boundary; with the physically inherited reservoir (above)
it does not exist.

## Summary

* `mass_flow_outlet` (pinned outflow, floating pressure) and `choked_nozzle_outlet` (compact
  sonic throat, critical mass flux) join `pressure_outlet` as **outflow-only** boundaries.
* Their acoustic closures — constant mass flow $\dot m'=0$ and Marble–Candel — together with the
  total-pressure inlet's convective open end $-\tfrac{1-M}{1+M}$, are **read out of $J_\text{alg}$**
  to round-off (§3); no separate boundary condition is specified.
* The build-time check rejects boundary sets with **no pressure reference** (§2).
* **Acoustic-power diagnostics** (`nefes.perturbation.fields.power`): with mean flow the energy-neutral
  reflection magnitude is $\frac{1+M}{1-M}$ (outlet) / $\frac{1-M}{1+M}$ (inlet). The inherited
  reservoir lands exactly on the inlet bound — it is **energy-neutral** — so the rig is passive
  and every mode decays (§5). Forcing the inlet above its bound makes it an acoustic source and
  manufactures a spurious instability, which `result.boundary_power(i)` pins squarely on the
  inlet ($2\sigma E = \sum W_\text{in}$).

## Export for FNetLibUI

The full network is available as `network` (a `Network`) and its converged mean flow as `sol` (a `Solution`).
Save either to a UI-readable YAML — `sol.to_yaml(path)` embeds the mean-flow result fields, `network.to_yaml(path)` writes the topology only — then open the file in FNetLibUI.

In [ ]:
import os, tempfile

sol = network.solve()
_out = os.path.join(tempfile.mkdtemp(), "outlet_boundaries.yaml")
sol.to_yaml(_out)  # embeds the mean-flow results; use network.to_yaml(_out) for topology only
print("saved case:", _out)